# Módulo AI6 · Alineación — Práctica (CPU, todo determinista)

1. **Modelo de recompensa (RM)**: pares preferido/rechazado → regresión logística
2. **DPO** con política de 2 acciones: la pérdida completa, derivada y optimizada a mano
3. **GRPO** por grupos con la misma política
4. Registro de experimentos (seed, datos, config)


In [ ]:
import json, math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(42); np.random.seed(42)

## 1 · Dataset de preferencias sintético (juguete)

Cada ejemplo: instrucción → 2 respuestas candidatas con 3 features de calidad (relevancia, brevedad, tono). Los humanos marcaron cuál prefieren.

In [ ]:
# features = [relevancia(0-1), brevedad(0-1), tono(0-1)]
PREFS = [
    {"a": [0.9, 0.7, 0.9], "b": [0.7, 0.4, 0.5], "pref": "a"},  # a más útil
    {"a": [0.8, 0.2, 0.9], "b": [0.7, 0.9, 0.6], "pref": "a"},  # a útil > concisa
    {"a": [0.4, 0.9, 0.8], "b": [0.8, 0.9, 0.3], "pref": "b"},  # b más útil
    {"a": [0.6, 0.9, 0.9], "b": [0.6, 0.8, 0.4], "pref": "a"},  # tono manda
    {"a": [0.9, 0.5, 0.5], "b": [0.5, 0.9, 0.5], "pref": "a"},
    {"a": [0.3, 0.4, 0.9], "b": [0.8, 0.6, 0.6], "pref": "b"},
    {"a": [0.7, 0.6, 0.7], "b": [0.9, 0.6, 0.2], "pref": "a"},
    {"a": [0.5, 0.9, 0.4], "b": [0.5, 0.9, 0.9], "pref": "b"},
]
len(PREFS)

## 1b · Entrenar el modelo de recompensa (RM)

RM: recompensa r(x) = w·features. Se entrena minimizando la log-pérdida de preferencia:
P(pref=a) = σ(r(a) − r(b))

In [ ]:
def train_rm(prefs, pasos=400, lr=0.1):
    w = torch.zeros(3, requires_grad=True)
    opt = torch.optim.Adam([w], lr=lr)
    for paso in range(pasos):
        e = prefs[paso % len(prefs)]
        ra = (w * torch.tensor(e["a"])).sum()
        rb = (w * torch.tensor(e["b"])).sum()
        p_a = torch.sigmoid(ra - rb)
        # pérdida: -log P(pref correcto)
        loss = -torch.log(p_a if e["pref"] == "a" else 1 - p_a + 1e-9)
        opt.zero_grad(); loss.backward(); opt.step()
    return w.detach()

w = train_rm(PREFS)
print("pesos RM [relevancia, brevedad, tono]:", np.round(w.numpy(), 2))
print("→ el RM aprendió qué importa a los humanos del dataset")

## 2 · DPO con política de 2 acciones

La política decide entre acción `a` y `b`. La pérdida DPO:

`L_DPO = −log σ( β [(log π(y_w) − log π_ref(y_w)) − (log π(y_l) − log π_ref(y_l))] )`


In [ ]:
beta = 2.0
logits = torch.tensor([0.0, 0.0], requires_grad=True)   # π (aprendemos estos scores)
logits_ref = torch.tensor([0.0, 0.0])                    # π_ref congelada

def dpo_paso(pref_idx, rechazado_idx, pasos=300, lr=0.5):
    opt = torch.optim.Adam([logits], lr=lr)
    for paso in range(pasos):
        logp_pi_w = F.log_softmax(logits, 0)[pref_idx]
        logp_ref_w = F.log_softmax(logits_ref, 0)[pref_idx]
        logp_pi_l = F.log_softmax(logits, 0)[rechazado_idx]
        logp_ref_l = F.log_softmax(logits_ref, 0)[rechazado_idx]
        loss = -F.logsigmoid(beta * ((logp_pi_w - logp_ref_w) - (logp_pi_l - logp_ref_l)))
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

# con las features del RM: la respuesta preferida tiene mayor recompensa RM
e = PREFS[0]
idx_a, idx_b = 0, 1
pref_idx = idx_a if e["pref"] == "a" else idx_b
l_dpo = dpo_paso(pref_idx, idx_b if pref_idx == idx_a else idx_a)
p = F.softmax(logits, 0).detach().numpy()
print(f"pérdida DPO final: {l_dpo:.3f}")
print(f"π(a)={p[0]:.2f}  π(b)={p[1]:.2f}  → la política manda a la respuesta preferida")

## 3 · GRPO: recompensa por grupo

El grupo = k respuestas a una misma instrucción. Recompensa normalizada *dentro del grupo*; se refuerza las que superan la media. Sin value model, como PPO.


In [ ]:
def grpo_grupo(scores, grupo, pasos=300, lr=0.5):
    """scores: recompensa RM de cada acción; grupo: acciones que componen el grupo."""
    media = np.mean([scores[i] for i in grupo])
    ventajas = {i: scores[i] - media for i in grupo}
    logits_g = torch.tensor([0.0, 0.0], requires_grad=True)
    opt = torch.optim.Adam([logits_g], lr=lr)
    for paso in range(pasos):
        logp = F.log_softmax(logits_g, 0)
        # ventaja como recompensa: se refuerza proporcional a (score - media)
        loss = -(sum((v * logp[i]) for i, v in ventajas.items() if abs(v) > 1e-6))
        opt.zero_grad(); loss.backward(); opt.step()
    return F.softmax(logits_g, 0).detach().numpy(), float(loss)

e = PREFS[0]
scores = {(w * torch.tensor(e["a"])).sum().item(), (w * torch.tensor(e["b"])).sum().item()}
scores = [(w * torch.tensor(e["a"])).sum().item(), (w * torch.tensor(e["b"])).sum().item()]
p_grpo, perdida = grpo_grupo(scores, [0, 1])
print(f"GRPO: π={np.round(p_grpo,2)}  (la acción con más recompensa RM gana)")
print("→ RL por grupos: cada acción se refuerza según su ventaja dentro del grupo")

## 4 · Registro de experimentos (fair A/B con AI4)

La alineación sin registro es una opinión. Cada experimento: seed + dataset (versión) + β/lr + judge usado.


In [ ]:
REGISTRO = {}
for beta in (0.5, 2.0, 8.0):
    logits_e = torch.tensor([0.0, 0.0], requires_grad=True)
    logits_ref_e = torch.tensor([0.0, 0.0])
    opt = torch.optim.Adam([logits_e], lr=0.5)
    for paso in range(300):
        pref_idx = 0
        logp_w = F.log_softmax(logits_e, 0)[0] - F.log_softmax(logits_ref_e, 0)[0]
        logp_l = F.log_softmax(logits_e, 0)[1] - F.log_softmax(logits_ref_e, 0)[1]
        loss = -F.logsigmoid(beta * (logp_w - logp_l))
        opt.zero_grad(); loss.backward(); opt.step()
    p = F.softmax(logits_e, 0).detach().numpy()
    REGISTRO[beta] = {"p_preferida": float(p[0]), "loss": float(loss)}
pd = REGISTRO
import pandas as _pd
print(_pd.DataFrame(REGISTRO).T.round(3))
print("→ β alto: converge rápido pero puede sobre-castigar (tibio); β bajo: lentoo")

---

## Resumen

- **RM** = qué prefieren los humanos → pesos aprendidos de los pares.
- **DPO**: optimiza la preferencia *directamente* contra π_ref (sin RL explícito); β controla agresividad.
- **GRPO**: recompensas normalizadas por grupo → menos cómputo que PPO.
- **Registro** + eval (AI4) = la alineación se decide con datos, no con vibes.

## Ejercicios (4)
1. Cambia 2 preferencias del dataset (invirte "pref"). ¿Cómo cambian los pesos del RM? ¿Qué implica para datos reales?
2. Entrena DPO con β=0.1 y β=20. ¿Cuál produce respuestas "tibias" (probabilidad cercana a 0.5)? ¿Por qué?
3. En GRPO, usa grupos con 4 acciones (añade 2 más con scores intermedios). ¿Cómo cambia la ventaja?
4. *(Colab)* Lee el DPOTrainer de TRL y escribe el esqueleto con tu dataset de 8 pares (sin ejecutar).
